In [1]:
# %%



#------------------------------------------------ Begin_Librairie ----------------------------------------



from bs4 import BeautifulSoup

import datetime

import pandas as pd

from pandas import ExcelWriter

from selenium import webdriver

from selenium.webdriver.common.by import By

from webdriver_manager.chrome import ChromeDriverManager

# import bvdpdf

from time import sleep

import os

from selenium.common.exceptions import NoSuchElementException

from selenium.common.exceptions import ElementClickInterceptedException

from selenium.webdriver.support import expected_conditions as EC

import re

import requests

from bs4 import BeautifulSoup

import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)



# %%



#------------------------------------------------ Begin_ fileName ----------------------------------------



print("Running KY CIMA Web Scraping Tool v.1.0")



now=datetime.datetime.now()

filename= 'KY CIMA SQL Ready {}.xlsx'.format(str(now).replace(":",".")[:-7])

regulatorName = 'KY CIM'

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder = os.path.dirname(os.path.abspath(__file__))

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')



if os.path.exists(tempfolder):

    for rem_file in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem_file))

else:

    os.mkdir(tempfolder)

# %%



#------------------------------------------------ Begin_chromedriver ----------------------------------------



chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

chromeOptions.add_experimental_option("excludeSwitches", ["enable-automation"])

user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"

chromeOptions.add_argument(f"user-agent={user_agent}")



driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()

#------------------------------------------------ Begin_Fouction ----------------------------------------



def bourange_same_length_array(sqldict) :



    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])



            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict

#------------------------------------------------ Begin_Varible ----------------------------------------



sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 

		  'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 

		  'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 

		  'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],

		  'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 

		  'Phone - Mother company': [], 'Check': []}



processdate=now.strftime('%Y-%m-%d')

# %%



#------------------------------------------------ Begin_Variable ----------------------------------------



regdict = { 



            'KY CIM 1':'Banking Class A', 



            'KY CIM 2':'Banking Class B', 



            'KY CIM 3':'Money Services', 



            'KY CIM 4':'Trust (Registered PTC)', 



            'KY CIM 5':'Trust (Restricted)',

            'KY CIM 6':'Nominee (Trust)', 



            'KY CIM 7':'Trust (Controlled Subsidiary)', 



            'KY CIM 8':'-', 



            'KY CIM 9':'Company Manager', 



            'KY CIM 10':'Corporate Service Provider', 

            'KY CIM 11':'Full List of all Insurance Entities Registered with the Cayman Islands Monetary Authority', 

            'KY CIM 12':'List of all Mutual Funds registered/licensed with the Cayman Islands Monetary Authority', 



            'KY CIM 13':'List of all Mutual Fund Administrators licensed with the Cayman Islands Monetary Authority', 



            'KY CIM 14':'Securities - Registered Person', 



            'KY CIM 15':'Private Fund', 



            'KY CIM 16':'Virtual Asset Service Provider Registration', 



            'KY CIM 17':'Securities - Full', 



            'KY CIM 18':'Building Society',

            'KY CIM 19':'Credit Union',



            'KY CIM 20':'Development Bank', 





            }





mapping = {
    "Banking Class A": ("1", "Banking Class A"),
    "Banking Class B": ("2", "Banking Class B"),
    "Money Services": ("3", "Money Services"),
    "Trust": ("4", "Trust"),
    "Trust (Restricted)": ("5", "Trust (Restricted)"),
    # also regulated type is regulated?
    "Nominee (Trust)": ("6", "Nominee (Trust)"),
    "Trust (Controlled Subsidiary)": ("7", "Trust (Controlled Subsidiary)"),
    "Trust (Registered PTC)": ("8", "Trust (Registered PTC)"),
    "Company Manager": ("9", "Company Manager"),
    "Corporate Service Provider": ("10", "Corporate Service Provider"),
    
    "Class A Local Insurer":("11","Insurance Entities"),
    "Class A External Insurer":("11","Insurance Entities"),
    "Class B Insurer":("11","Insurance Entities"),
    "Class C Insurer":("11","Insurance Entities"),
    "Class D Insurer":("11","Insurance Entities"),
    "Insurance Agent":("11","Insurance Entities"),
    "Insurance Broker":("11","Insurance Entities"),
    "Insurance Manager":("11","Insurance Entities"),
    "Portfolio Insurance Company":("11","Insurance Entities"),
    
    "Mutual Fund - Registered": ("12", "Mutual Fund - Registered/Licenced"),
    "Mutual Fund - Licenced": ("12", "Mutual Fund - Registered/Licenced"),
    "Mutual Fund - Administered": ("13", "Mutual Fund - Administered"),
    
    "Securities - Registered Person": ("14", "Securities - Registered Person"),
    "Private Fund": ("15", "Private Fund"),
    "Virtual Asset Service Provider Registration": ("16", "Virtual Asset Service Provider Registration"),
    "Securities - Full": ("17", "Securities - Full"),
    "Building Society": ("18", "Building Society"),
    "Credit Union": ("19", "Credit Union"),
    "Development Bank": ("20", "Development Bank"),
}



#------------------------------------------------ Main_Function ----------------------------------------




url = "https://www.cima.ky/search-entities-cima/get_search_data"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7",
    "Accept-Language": "en-US,en;q=0.9",
    "Content-Type": "application/x-www-form-urlencoded",
    "Origin": "https://www.cima.ky",
    "Referer": "https://www.cima.ky/search-entities-cima",
}
cima_cfrf_token_cookie_name = "b49978775cd9ff422a168224b35d4da5"
cima_cfrf_token_name = 'b49978775cd9ff422a168224b35d4da5'
phpsessid = "78447b15c86cc83e2e6f5f6981aab1e7"
g_recaptcha_response = "0cAFcWeA4733Aw1KnwapFICQqhFV0yVXIOO-e4XzegW4awz1oTHA8rcOPwjDAdnclBbQcSYr3WBOYZt6B6i0CX9IzYBFrMqG_X7Em9PG1ko9k7_jccgYMD5Tk7hdkmvaiDFwdWBFfNAEd0JOOYHtA_B2LOwbHumP3xnNKmVc3uwR8II-0FeuHmfIZitK2rQsGi5TyDA04XO0G_Zp-ZuBReUEA7IYykpTnGQkZmUsk2TH-H32Pc-YC8pzZ4XomleoHA0U0ZB-9e66p4ifDq5xrsndENs5LmaO9qbqJcS2zgz_-Ij3cT1N9rU_wwoTHR-VvrGz1PeTXIC4OXSo0a61qVfHpG-2bLoRofUutCvT6BJXKKRHpaYWbbXtKPrPEj5iIMmDI7R-Ui-bjjHrGGInhYb9XgNYiPyQZpm46tOlvODmUsovT1OdV2OQNOCIMTjUhFVyZ2HkGmtfhVkAIJlJ_-WCrWqpmsjvxkGUagHkLZa0S3k9wvsMfx7syYJVXLd_Ui_MebKZLz9yDym8M1JlZUHq3sp1x2arVboH6cP57zB-eo3DYTLRFgp8mDh5oo24xwPKdTA2A_9ERSpwO5oQgOUiyXkDTi_gLjFB6Km0RDYfOo_CZlCM92Rghbz1FVYPPdFPbwlM87jhzRNdR0tqmToT0_r-K6QOIjmyNyNr3McTQ0OAWucWM79q6dM5cmUW1YR9qerwfsCxnEZErkm3V9sCfglOCN_krsPrBxUaFdexu3QlU_z_mSsBErNdcFDlGW8JuUEVcWz2j3LzMaOIi35sTCq0FdM48IXD2RzOPC5000VKdQ9QjngIBR4NMXsaRBOk6ph1KNiN6rwW3tI3Zdz36pIiFaD52zs5Ap4IRcL4Y3F6l7KxSr5oDcMlDrKSd_V3k9xsL3i9NEmWG4LcrkNtUOeKJaRZi4d17NtOD9GYaUZ78dmYetuTDs2iwyOyAWBSKOXrAM8cSud0_pZmCoGr5vh-KT8YK5jQLUbD7lVjy_go6DhJ5B_OChXFmztB4dRYXsEOf4LAOONaOvfrK6Pc5_xClWwtCLKH7w1t3E1CVrxT-NGEVUH3ydedGOeKKQovzCOJpqGnA4be1lhamUn3uFCvHFN3JdRdn2kQkjP6e-Xz4dgreEEPaj71EeYvwF1xEKw90fd8UpaKeMWERMlInPIWPPxoA0oIqdAtW8lAnz7LzOdPjVULi2-DM-uRWRiHaPsEe4G5kJKoUEKSdWtGo3NBRyWmhJAIbR2tghgI-pAc3KGy_zyvsxj_6LHPaXEUnCDCVeRWNqkuTQ_py35WB-KLzUJ2BGRcPwJjxdASNd8cRj5Br2Bklh2ZUwmgsJwPrG45_U2Cme5m1kqEF0UjqI4jXrME64lEToqevlC-8n9mKw6ySX8SajiRzxcsH7o82mAhw1t381ruOTCVKQbdRgtpvP_3D6H6yIMiK3HmT4LouGIDh6apm8LthbtvjuXDW1JziTgRQYF27OpVKfrAVzPJVnrTqPA6znR1gp1sqIIcCVWwgRHSzgBSfucfS0VWleHsEGQH1027-1SHwHCPBM9ZhVeA3EAU0Z-F3OjU7k2TgcQSTAdT54vboiZdI-dlzko5e-270q43x-KkIPeHR6h37uR75K5vh6j1uC23wvkUGjagOYup3f86USrx2LvIFbFj_0UbKdGLFxsI9cOMJpZO5EoNOgk9p1ppWzzjqQmCbRowddVUrcnsWMvwa2HOzN8Cjs-PIujMJRchRxZEneurOnBPHEbuucuZnAleGnaSynSaF5r7h6e4zOh1fSfubIj37etHmz"


cookies = {
    "cima_cfrf_token_cookie_name": cima_cfrf_token_cookie_name,
    "PHPSESSID": phpsessid,
    "popup": "Y",
}

session = requests.Session()

# token keep 24h alive, change everytime
base_payload = {
    "cima_cfrf_token_name": cima_cfrf_token_name,
    "Searching": "",
    "AuthorizationType": "All",
    #'AuthorizationType':'Trust_(Registered_PTC)',
    "g-recaptcha-response": g_recaptcha_response,
    "hiddenRecaptcha": "",  # include if the form has it
    "p": "y",
    "ajax": "Y",
}


response = session.post(url, data=base_payload, headers=headers, cookies=cookies, timeout=30,verify=False)





if response.status_code != 200:

    raise RuntimeError(f"Page 1 failed: HTTP {response.status_code}")



# --- First Page ---

soup = BeautifulSoup(response.text, "html.parser")

table = soup.select_one("div.table-responsive")

rows = table.select("tr") if table else []

print(f"Fetching 1 page...")

for row in rows[1:]:

    cells = [td.get_text(strip=True) for td in row.find_all("td")]

    internal_id = cells[0]

    name_ = cells[1]

    type_ = cells[2]

    regulated_date = cells[3]

    sqldict['Name'].append(name_)

    sqldict['Typology'].append(type_)

    sqldict['InternalID_1'].append(internal_id)

    sqldict['InternalID_1_type'].append('Reference Number')

    sqldict['ListProcessDate'].append(processdate)



    sqldict['RegulationDate'].append(regulated_date)

    #sqldict['ListName'].append(Typology[reg])

    sqldict['RegulationType'].append('Regulated')

    # sqldict['RegCtry'].append(reg.split(' ')[0])

    # sqldict['RegCode'].append(reg.split(' ')[1])

    # sqldict['ListCode'].append(reg.split(' ')[-1])

sqldict = bourange_same_length_array(sqldict)



# --- The rest pages ---

while True:

    page_list = soup.find_all("div", class_="pagination-wrap")

    next_li = soup.select_one("li#last a.last")

    if not next_li:

        print("Reached final page.")

        break

    onclick = next_li.get("onclick", "")

    match = re.search(r"PageNumber=(\d+)", onclick)

    if not match:

        print("Next link missing PageNumber; stopping.")

        break

    next_page = int(match.group(1))

    print(f"Fetching page {next_page}...")

    payload = {
            "PageNumber": next_page,
            "p": 'y',
            "ajax": "Y",
            "cima_cfrf_token_name": cima_cfrf_token_name,
            "Searching": "",
            "AuthorizationType": "All",
            }

    resp = session.post(url, data=payload, headers=headers, cookies=cookies, timeout=30, verify=False)

    if resp.status_code != 200:

        raise RuntimeError(f"Page {next_page} failed: HTTP {resp.status_code}")



    soup = BeautifulSoup(resp.text, "html.parser")

    table = soup.select_one("div.table-responsive")

    rows = table.select("tr") if table else []



    for row in rows[1:]:

        cells = [td.get_text(strip=True) for td in row.find_all("td")]

        internal_id = cells[0]

        name_ = cells[1]

        type_ = cells[2]

        regulated_date = cells[3]

        sqldict['Name'].append(name_)

        sqldict['Typology'].append(type_)

        sqldict['InternalID_1'].append(internal_id)

        sqldict['InternalID_1_type'].append('Reference Number')

        sqldict['ListProcessDate'].append(processdate)



        sqldict['RegulationDate'].append(regulated_date)

        #sqldict['ListName'].append(Typology[reg])

        sqldict['RegulationType'].append('Regulated')

        # sqldict['RegCtry'].append(reg.split(' ')[0])

        # sqldict['RegCode'].append(reg.split(' ')[1])

        # sqldict['ListCode'].append(reg.split(' ')[-1])

    sqldict = bourange_same_length_array(sqldict)



# %%



#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

df=pd.DataFrame(sqldict)

mapped = df["Typology"].map(mapping).apply(pd.Series)
mapped.columns = ["ListCode", "ListName"]
df[["ListCode", "ListName"]] = mapped
df['ListProcessDate'] = processdate
df['RegCtry'] = 'KY'
df['RegCode'] = 'CIMA'
df['RegulationType'] = df['RegulationType'].fillna('Regulated')
df = (
    df.assign(ListCode=df['ListCode'].fillna('').astype(str).str.strip())
      .loc[lambda d: d['ListCode'] != '']
)
df.to_excel(filename, index=False)

sleep(3)

driver.quit()

    
    
    
    
    
    

Running KY CIMA Web Scraping Tool v.1.0
Fetching 1 page...
Fetching page 2...
Fetching page 3...
Fetching page 4...
Fetching page 5...
Fetching page 6...
Fetching page 7...
Fetching page 8...
Fetching page 9...
Fetching page 10...
Fetching page 11...
Fetching page 12...
Fetching page 13...
Fetching page 14...
Fetching page 15...
Fetching page 16...
Fetching page 17...
Fetching page 18...
Fetching page 19...
Fetching page 20...
Fetching page 21...
Fetching page 22...
Fetching page 23...
Fetching page 24...
Fetching page 25...
Fetching page 26...
Fetching page 27...
Fetching page 28...
Fetching page 29...
Fetching page 30...
Fetching page 31...
Fetching page 32...
Fetching page 33...
Fetching page 34...
Fetching page 35...
Fetching page 36...
Fetching page 37...
Fetching page 38...
Fetching page 39...
Fetching page 40...
Fetching page 41...
Fetching page 42...
Fetching page 43...
Fetching page 44...
Fetching page 45...
Fetching page 46...
Fetching page 47...
Fetching page 48...
Fetching 